Baseline comparison with model $K_l^{8/5}=649+3.76/R_{nd}^{0.63}$

# Libraries

In [1]:
import os
from pathlib import Path
import numpy as np

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
RANDOM_STATE = 42

In [3]:
# train/test split
import sys
sys.path.insert(1, '../utils_functionality/split_utils/')
from split_tools import get_train_test

In [4]:
# dict with num features
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_hyperparams import dict_num_features

# Selected models metrics

In [5]:
df_metrics = pd.DataFrame()
path_models = Path('../results/models_modelling_2/')

In [6]:
def calc_current_metrics(target, model, fnames=False):
    path_pipeline = path_models / model
    pipeline = joblib.load(path_pipeline)
    _, test = get_train_test(
        dataset_filename='df_modelling_dimensionless',
        target=target)
    if fnames: test = test[pipeline.steps[0][1].feature_names_+[target]]
    else: test = test[dict_num_features['df_modelling_dimensionless']\
                    +['wettability']+[target]]
    preds = pipeline.predict(test.drop(columns=[target]))
    df_current_metrics = pd.DataFrame({
            'model': [model],
            'target': [target],
            'accuracy': [accuracy_score(test[target], preds)],
            'f1': [f1_score(test[target], preds)],
            'precision': [precision_score(test[target], preds)],
            'recall': [recall_score(test[target], preds)],
            'roc_auc': [roc_auc_score(test[target], preds)]})
    return df_current_metrics

../results/models_modelling_2/logisticregression_splashing_df_modelling_dimensionless_ordenc

In [7]:
df_current_metrics = calc_current_metrics(target='splashing', 
                                        model='logisticregression_splashing_df_modelling_dimensionless_ordenc')
df_metrics = pd.concat((df_current_metrics, df_metrics))

../results/models_modelling_2/logisticregression_net_impact_df_modelling_dimensionless_ordenc

In [8]:
df_current_metrics = calc_current_metrics(target='net_impact', 
                                        model='logisticregression_net_impact_df_modelling_dimensionless_ordenc')
df_metrics = pd.concat((df_current_metrics, df_metrics))

../results/models_modelling_2/catboostclassifier_splashing_df_modelling_dimensionless

In [9]:
df_current_metrics = calc_current_metrics(target='splashing', 
                                        model='catboostclassifier_splashing_df_modelling_dimensionless',
                                        fnames=True)
df_metrics = pd.concat((df_current_metrics, df_metrics))

../results/models_modelling_2/catboostclassifier_net_impact_df_modelling_dimensionless

In [10]:
df_current_metrics = calc_current_metrics(target='net_impact', 
                                        model='catboostclassifier_net_impact_df_modelling_dimensionless',
                                        fnames=True)
df_metrics = pd.concat((df_current_metrics, df_metrics))

In [11]:
df_metrics

,model,target,accuracy,f1,precision,recall,roc_auc
0,catboostclassifier_net_impact_df_modelling_dim...,net_impact,0.946667,0.900000,0.900000,0.9000,0.931818
0,catboostclassifier_splashing_df_modelling_dime...,splashing,0.866667,0.900000,0.865385,0.9375,0.839120
0,logisticregression_net_impact_df_modelling_dim...,net_impact,0.893333,0.800000,0.800000,0.8000,0.863636
0,logisticregression_splashing_df_modelling_dime...,splashing,0.813333,0.865385,0.803571,0.9375,0.765046


# Comparsion with decision stump, splashing

In [12]:
path_data = Path('../data')
df_main = pd.read_excel(path_data / 'df_main.xlsx')

In [13]:
target = 'splashing'
_, test = get_train_test(
    dataset_filename='df_modelling_dimensionless',
    target=target)

In [14]:
test_full = test.merge(df_main, left_index=True, right_index=True,
                 how='left', suffixes=('', '_y'))
test_full.drop(test_full.filter(regex='_y$').columns, axis=1, inplace=True)
test_full.head()

,splashing,wettability,inclination,roughness_binary,particle_liquid_density_ratio,volume_fraction_binary,particle_droplet_diameter_ratio,Re,We,We_Re,...,liquid_density,surface_tension,viscosity,particle_mean_diameter,particle_density,volume_fraction,droplet_diameter,height,particle_diameter_cat,velocity
291,1,2,45,0,1.016949,1,0.078797,706.180601,951.652336,159.025989,...,1180,0.0679,0.02310,0.000275,1200,0.08,0.00349,0.8,large,3.961141
328,0,0,0,0,1.016949,1,0.013259,950.005136,1920.347158,243.288377,...,1180,0.0679,0.02310,0.000041,1200,0.08,0.00313,1.8,small,5.941712
4,1,1,0,0,1.219512,1,0.013833,1435.111557,1434.906112,233.148786,...,820,0.0269,0.00679,0.000041,1000,0.08,0.00300,0.8,small,3.961141
313,0,1,45,0,0.847458,1,0.016339,256.976895,173.151643,52.684981,...,1180,0.0679,0.02310,0.000041,1000,0.10,0.00254,0.2,small,1.980571
175,1,1,0,0,1.000000,1,0.011891,13292.675924,748.091989,293.684216,...,1000,0.0732,0.00104,0.000041,1000,0.08,0.00349,0.8,small,3.961141


In [15]:
test_full['kl'] = (649 + 3.76 / (
    (test_full['roughness'] / (test_full['droplet_diameter']*(10**6))) ** 0.63)) ** (5/8)
preds = test_full['We_Re'] > test_full['kl']
df_current_metrics = pd.DataFrame({
        'model': ['decision_stump_baseline'],
        'target': [target],
        'accuracy': [accuracy_score(test[target], preds)],
        'f1': [f1_score(test[target], preds)],
        'precision': [precision_score(test[target], preds)],
        'recall': [recall_score(test[target], preds)],
        'roc_auc': [roc_auc_score(test[target], preds)]})
df_metrics = pd.concat((df_current_metrics, df_metrics))
df_metrics

,model,target,accuracy,f1,precision,recall,roc_auc
0,decision_stump_baseline,splashing,0.760000,0.800000,0.857143,0.7500,0.763889
0,catboostclassifier_net_impact_df_modelling_dim...,net_impact,0.946667,0.900000,0.900000,0.9000,0.931818
0,catboostclassifier_splashing_df_modelling_dime...,splashing,0.866667,0.900000,0.865385,0.9375,0.839120
0,logisticregression_net_impact_df_modelling_dim...,net_impact,0.893333,0.800000,0.800000,0.8000,0.863636
0,logisticregression_splashing_df_modelling_dime...,splashing,0.813333,0.865385,0.803571,0.9375,0.765046


In [16]:
df_metrics[df_metrics['target']=='splashing']

,model,target,accuracy,f1,precision,recall,roc_auc
0,decision_stump_baseline,splashing,0.760000,0.800000,0.857143,0.7500,0.763889
0,catboostclassifier_splashing_df_modelling_dime...,splashing,0.866667,0.900000,0.865385,0.9375,0.839120
0,logisticregression_splashing_df_modelling_dime...,splashing,0.813333,0.865385,0.803571,0.9375,0.765046
